In [1]:
%load_ext autoreload
%autoreload 2


In [2]:
# base imports
import numpy as np 
import pandas as pd
from functools import partial
import pickle
import warnings
import os

# preprocessing
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from bs4 import BeautifulSoup

import re

# NLP packages
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.model_selection import train_test_split
import scipy

from datasets import load_dataset

max_length = 32
# size = 'all'
size = 250
"""
Data Preprocessing Functions
"""
stemmer = PorterStemmer()
lemma = WordNetLemmatizer()

def tokenize(text,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    tokens = [word for word in text.lower() if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def token2index(tokens,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def assert_even_length_array(arr):
    if len(arr) % 2 != 0:
        raise AssertionError("Array does not have an even number of elements: {}".format(len(arr)))

def check_order(array):
    prev_entry = None
    ilist = []  
    for i, entry in enumerate(array):
        if entry == prev_entry:
            ilist.append(i)
    
        prev_entry = entry

    return ilist

def sort_data_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array.iloc[i] = temp2
        array.iloc[i+1] = temp1

    return array

def sort_label_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array[i] = temp2
        array[i+1] = temp1

    return array


def get_vector(vec1,vec2):
    
    d_vec = vec2 - vec1
    
    if np.sum(d_vec) == 0:
        
        warnings.warn("A text vector and its' counterfactual vector have the same value. This is probably not right.")
        
    if isinstance(d_vec, scipy.sparse.csr_matrix):  # or any other sparse matrix type
        mag = scipy.linalg.norm(d_vec.todense())

    elif isinstance(d_vec, np.ndarray):
        mag = np.linalg.norm(d_vec)
        # Perform operations specific to dense arrays
    else:
        print("Unknown vector type")
        mag = np.linalg.norm(d_vec)
    
    if mag!=0:
        return d_vec/mag        
    else:
        return np.zeros((np.shape(vec1)[0]))
    # return [d_vec/mag]



## cleaning the text

def cleantext(text):
    # removing the "\"
    text = re.sub("'\''","",text)
    
    # removing special symbols
    text = re.sub("[^ a-zA-Z]","",text)
    
    # removing the whitespaces
    text = ' '.join(text.split())
    
    # convert text to lowercase    
    text = text.lower()
    
    return text

# removing the stopwords
stop_words = set(stopwords.words('english'))
mod_stop_words = set(stopwords.words('english')) - {'not', 'but'}
# stop_words = stop_words - set(['dont','do'])
# def removestopwords(text):
    
#     removedstopword = [word for word in text.split() if word not in mod_stop_words]
#     return ' '.join(removedstopword)

def removestopwords(text):
    
    removedstopword = [word for word in text if word not in mod_stop_words]
    return removedstopword

#Removing the html strips 
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

#Removing the square brackets
def remove_between_square_brackets(text):
    return re.sub(r'\[[^]]*\]', '', text)

#Removing Emails
def remove_Emails(text):
    pattern=r'\S+@\S+'
    text=re.sub(pattern,'',text)
    return text

#Removing URLS
def remove_URLS(text):
    pattern=r'http\S+'
    text=re.sub(pattern,'',text)
    return text

def remove_special_characters(text, remove_digits=True):
    pattern=r'[^a-zA-z0-9\s]'
    text=re.sub(pattern,' ',text)
    return text

#Removing numbers
def remove_numbers(text):
    pattern = r'\d+'
    text = re.sub(pattern, '', text)
    return text

def lematizing(sentence):
    stemSentence = ""
    for word in sentence.split():
        stem = lemma.lemmatize(word)
        stemSentence += stem
        stemSentence += " "
    stemSentence = stemSentence.strip()
    return stemSentence

def stemming(sentence):
    
    stemmed_sentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemmed_sentence+=stem
        stemmed_sentence+=" "
        
    stemmed_sentence = stemmed_sentence.strip()
    return stemmed_sentence



def process_data(data):
    # Step 1: clean text (string level)
    data = data.apply(lambda x: strip_html(x))
    data = data.apply(remove_between_square_brackets)
    data = data.apply(lambda x: cleantext(x))
    data = data.apply(remove_special_characters)
    data = data.apply(remove_URLS)
    data = data.apply(remove_Emails)
    data = data.apply(remove_numbers)
    
    # Step 2: tokenize
    data = data.apply(lambda x: word_tokenize(x.lower()))
    
    # Step 3: remove stopwords and stem (token level)
    data = data.apply(lambda tokens: [stemming(token) for token in removestopwords(tokens)])

    return data




def gen_knowledge(data,label,cf=False):
    """
    Takes sequences of pairs of counterfactuals and 
    returns just the originals, with the counterfactual 
    included as an annotation.

    If cf=True, returns the counterfactuals and omits the originals
    """
    
    vectors_in = data['vector']
    text_in = data['text']
    # sparse_array = sp.csr_matrix((0,dlength))

    text_out=[]
    vectors_out = []
    labels = []
    # vectors = np.empty((0,1,dlength))
    cf_text = []
    cf_labels = []
    cf_vectors = []
    
    for i in range(int(np.shape(vectors_in)[0]/2)):
        
        
        if cf:
            vec1,vec2 = vectors_in[i * 2 + 1],vectors_in[i * 2]  # Get the data from the current dataset slice
            label_out = label[i*2+1]
            text = text_in[i * 2 + 1]
            _cf_text=text_in[i * 2]
        else:
            vec1,vec2 = vectors_in[i * 2],vectors_in[i * 2 + 1]  # Get the data from the current dataset slice                
            label_out = label[i*2]
            text = text_in[i * 2]
            _cf_text = text_in[i * 2 + 1]
            
        text_out.append(text)
        labels.append(label_out)
        vectors_out.append(vec1)
    
        cf_vectors.append([vec2])
        cf_labels.append([1 - label_out])
        cf_text.append([_cf_text])
    
    return {'vector':vectors_out,'text':text_out},labels,cf_vectors, cf_labels,cf_text



def compile_K(data,label, paired=False,cf=False,int_out=False):
    
    if paired:
        
        if np.shape(data['vector'])[0]%2 != 0:
            raise ValueError("Can't generate paired counterfactuals with an uneven number of samples")
        
        data,label,vector,labels,cf_text = gen_knowledge(data,label,cf=cf)
        
        # labels = [1 - l for l in label]
        
    else:
        warnings.warn("Non-paired data just creates blanks atm")
        
        vector = [[] for _ in range(np.shape(data['vector'])[0])]
        cf_text = [[] for _ in range(np.shape(data['vector'])[0])]

        labels = [[]]*len(vector)
    
    # labels = np.expand_dims(labels, axis=1)

    magnitude = np.ones(len(vector))
    magnitude = np.expand_dims(magnitude, axis=1)
    

    if int_out:
        for i in range(np.shape(vector)[0]):
            vector[i] = vector[i].astype(int)

    n_samples = len(data['text'])
    print(f'Returning {n_samples} samples with {len(vector)} counterfactuals')
    
    return {'text':data['text'],
            'X':data['vector'],
            'Y':label,
            'K':{'vector':vector,'label':labels,'magnitude':magnitude, 'text':cf_text}}


def compile_ag(X,y,text,cf_X,cf_text):
    
    magnitude = np.ones(len(cf_X))
    magnitude = np.expand_dims(magnitude, axis=1)

    n_samples = len(text)

    print(f'Returning {n_samples} samples with {len(cf_text)} counterfactuals')
    
    return {'text':list(text),
            'X':X,
            'Y':list(y),
            'K':{'X':np.expand_dims(X,axis=1),
                 'Y':np.expand_dims(np.full_like(np.array(y),np.nan),axis=1),
                 'K':np.expand_dims(cf_X,axis=1),
                 'magnitude':np.expand_dims(magnitude,axis=1),
                 'text':np.expand_dims(cf_text,axis=1)}}




/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment-cli/cli_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("SetFit/ag_news")

agnews = pd.read_json('data/original/fizle_AG_News_Llama3-70B-Instruct.json')
agnews_all = pd.read_csv('data/original/ag_news.csv')
agnews['label'] = agnews_all['label'][:500]


# X_train, X_test, y_train, y_test = train_test_split(agnews['input'], agnews['label'], test_size=0.2, random_state=42)
# X_test, X_dev, y_test, y_dev = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
if size == 'all':
    X_train, y_train = agnews['input'], agnews['label']
else:

    # X_train, y_train = agnews['input'][:size], agnews['label'][:size]
    _, X_train, _, y_train = train_test_split(agnews['input'], agnews['label'], test_size=size/500, random_state=42)

X_test, X_dev, y_test, y_dev = train_test_split(agnews_all['text'][500:], agnews_all['label'][500:], test_size=0.5, random_state=42)

In [ ]:
print(f"Training set size: {len(X_train)}")
print(f"Development set size: {len(X_dev)}")
print(f"Test set size: {len(X_test)}")
print(f"\nTotal samples: {len(X_train) + len(X_dev) + len(X_test)}")

In [5]:
agnews['input'][90:100]

90    U.S. Brokers Cease-fire in Western Afghanistan...
91    Sneaky Credit Card Tactics Keep an eye on your...
92    Intel Delays Launch of Projection TV Chip In a...
93    Fund pessimism grows NEW YORK (CNN/Money) - Mo...
94    Kederis proclaims innocence Olympic champion K...
95    Eriksson doesn #39;t feel any extra pressure f...
96    Injured Heskey to miss England friendly NEWCAS...
97    Staples Profit Up, to Enter China Market  NEW ...
98    Delegation Is Delayed Before Reaching Najaf AG...
99    Consumer Prices Down, Industry Output Up  WASH...
Name: input, dtype: object

In [6]:
agnews['label'][90:100]

90    0
91    2
92    3
93    2
94    1
95    1
96    1
97    2
98    0
99    2
Name: label, dtype: int64

In [7]:
X_train_processed = process_data(X_train)
X_dev_processed = process_data(X_dev)
X_test_processed = process_data(X_test)

In [8]:
X_train_processed

361    [arson, attack, jewish, centr, pari, afp, afp,...
73     [front, line, aid, russia, industri, citi, nor...
374    [goalhappi, ajax, feyenoord, maintain, perfect...
155    [futur, doctor, cross, border, student, mount,...
104    [mr, downer, shoot, mouth, alexand, downer, th...
                             ...                        
103    [surviv, biotech, downturn, charli, traver, of...
81     [capac, crowd, beach, volleybal, rock, joint, ...
38     [iran, warn, missil, hit, anywher, israel, teh...
314    [ual, creditor, agre, day, extens, ual, unit, ...
167    [cox, commun, form, committe, advis, buyout, c...
Name: input, Length: 250, dtype: object

In [9]:
cf_train = agnews['counterfactual'][list(X_train.index)]
# cf_dev = agnews['counterfactual'][list(X_dev.index)]
# cf_test = agnews['counterfactual'][list(X_test.index)]


In [10]:

data_for_vectorizer = X_train_processed.apply(lambda tokens: " ".join(tokens))
vectorizer = CountVectorizer(max_features=20000)
vectorizer.fit(data_for_vectorizer)
# Get the vocabulary mapping (word to integer index)
vocabulary = vectorizer.vocabulary_

# define a reusable lambda
to_indices = lambda texts: [token2index(text, vocabulary, max_length=max_length) for text in texts]

# now you can call it on any dataset
tfidf_train = to_indices(X_train_processed)
tfidf_dev   = to_indices(X_dev_processed)
tfidf_test  = to_indices(X_test_processed)

tfidf_train_cf = to_indices(process_data(cf_train))
# tfidf_dev_cf = to_indices(process_data(cf_dev))
# tfidf_test_cf = to_indices(process_data(cf_test))

In [11]:
tfidf_train_cf[0]

[np.int64(137),
 np.int64(152),
 np.int64(2258),
 np.int64(360),
 np.int64(1592),
 np.int64(41),
 np.int64(41),
 np.int64(2258),
 np.int64(2066),
 np.int64(360),
 np.int64(361),
 np.int64(1592),
 np.int64(587),
 np.int64(811),
 np.int64(1576),
 np.int64(137),
 np.int64(152),
 np.int64(406),
 np.int64(167),
 np.int64(1917),
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0]

In [12]:
y_train

361    0
73     0
374    1
155    0
104    0
      ..
103    2
81     1
38     0
314    2
167    2
Name: label, Length: 250, dtype: int64

In [13]:


"""
########################################################################################################################
Save embeddings
########################################################################################################################
"""
# Stripping html from unprocessed text, just to clean it up
print('\ntrain_Data')
cf_train={'original': compile_ag(np.array(tfidf_train),y_train,X_train,tfidf_train_cf,cf_train)} # X,y,text,cf_X,cf_text

for key,val in cf_train['original']['K'].items():
    print(key,np.shape(val))
    print(val[0:10])

print('\ndev_Data')
cf_dev={'original':{ 
                    'text':X_dev,
                    'X':tfidf_dev,
                    'Y':y_dev,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}

print('\ntest_Data')
cf_test={'original':{ 
                    'text':X_test,
                    'X':tfidf_test,
                    'Y':y_test,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}
   
print('\ncontrol_Data')
flattened_text=[t for ts in cf_train['original']['K']['text'] for t in ts]

flattened_X = [x for xs in cf_train['original']['K']['X'] for x in xs]
flattened_Y = [y for ys in cf_train['original']['K']['Y'] for y in ys]


cf_control={'original': {
            'text':flattened_text,
            'X':np.array(flattened_X),
            'Y':list(flattened_Y),
            'K':{
                'text':cf_train['original']['text'],
                'X':np.array(np.zeros_like(flattened_X)),
                'Y':np.array(np.zeros_like(flattened_Y)),
                'K':np.array(np.zeros_like(flattened_X)),
                'magnitude':np.array(np.zeros_like(flattened_X)),
                }}}
    

pickle_data = {'train':cf_train,
               'test':cf_test,
               'dev':cf_dev,
               'control':cf_control,
               'n_classes':len(np.unique(y_test))}

embedding_path = f'data/integer_len{max_length}_SIZE_{size}_noleaks.pkl'
print(f"Saving to {embedding_path}")
with open(embedding_path, 'wb') as file:
    pickle.dump(pickle_data, file)



train_Data
Returning 250 samples with 250 counterfactuals
X (250, 1, 32)
[[[ 137  152 1172  360 1592   41   41 1172 2066  360  361 1592  587  811
   1576  108  137  152  406  167 1917    0    0    0    0    0    0    0
      0    0    0    0]]

 [[ 878 1289   50 1910 1094  406 1518 1439 2174   50 1025  291 1675    0
      0    0    0    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[ 928   57  796 1337 1628 2135  369   57   89  321  216 2285 1471  279
   2197  796 1025  859 1604 2489 1069 2299 1817  661 1258  658  814  620
      0    0    0    0]]

 [[ 884  623  519  260 2175 1446 2039 1946 1372 1264  597  526 1997  999
    669  984    0    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[1452  635 2014 1448   61  635 2281  564 1783 1244  874 1777  499  811
   1414 1516 1223 2223 1770 2363 1749 1831  549  135 2539 1958 2429 1516
   1224    0    0    0]]

 [[2074  765    8  913 1000  859 2395 2074  372    9 1140 1717 199

/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment-cli/cli_venv/lib/python3.10/site-packages/numpy/_core/numeric.py:442: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


creating baseline with counterfactuals

In [14]:
cf_train['original']['text'][0:2]

['Arson attack on Jewish centre in Paris (AFP) AFP - A Jewish social centre in central Paris was destroyed by fire overnight in an anti-Semitic arson attack, city authorities said.',
 'On front line of AIDS in Russia An industrial city northwest of Moscow struggles as AIDS hits a broader population.']

In [15]:
print(cf_train['original']['X'][0:2])
print(cf_train['original']['Y'][0:2])


[[ 137  152 1172  360 1592   41   41 1172 2066  360  361 1592  587  811
  1576  108  137  152  406  167 1917    0    0    0    0    0    0    0
     0    0    0    0]
 [ 878 1289   50 1910 1094  406 1518 1439 2174   50 1025  291 1675    0
     0    0    0    0    0    0    0    0    0    0    0    0    0    0
     0    0    0    0]]
[0, 0]


In [16]:
print(cf_train['original']['K']['text'][0:2])
print(cf_train['original']['K']['K'][0:2])
print(cf_train['original']['K']['magnitude'][0:2])
print(cf_train['original']['K']['Y'][0:2])
print(cf_train['original']['K']['X'][0:2])

[['Arson attack on tennis centre in Paris (AFP) AFP - A tennis social centre in central Paris was destroyed by fire overnight in an arson attack, city authorities said.']
 ['On front line of doping in Russia An industrial city northwest of Moscow struggles as doping hits a broader population.']]
[[[ 137  152 2258  360 1592   41   41 2258 2066  360  361 1592  587  811
   1576  137  152  406  167 1917    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[ 878 1289  630 1910 1094  406 1518 1439 2174  630 1025  291 1675    0
      0    0    0    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]]
[[[1.]]

 [[1.]]]
[[0]
 [0]]
[[[ 137  152 1172  360 1592   41   41 1172 2066  360  361 1592  587  811
   1576  108  137  152  406  167 1917    0    0    0    0    0    0    0
      0    0    0    0]]

 [[ 878 1289   50 1910 1094  406 1518 1439 2174   50 1025  291 1675    0
      0    0    0    0    0    0    0    0    0    0    0    0    0    0
      0    0